# Robotics arena MuJoCo runtime checks
Executed in Google Colab on September 9, 2026 with an NVIDIA A100-SXM4-40GB. Native physics runs on the CPU; NVIDIA EGL rendering is verified separately.

Select an A100 runtime and run cells in order. These cells install pinned dependencies, inspect adhesion compatibility, register the existing NVIDIA EGL library, and validate the public migration branch.

The last cell checks out final source commit `d8c20fd6d9a81cb34986356cd0b222109f94d0f5` and records all 21 tests passing, the native 0.02-second run, and the arena render. Earlier cells preserve the initial validation on `dc77f2aeae639e3120ece2857d74bcfb3bec24d5`.

Live notebook: https://colab.research.google.com/drive/1zmPS7IvWuNpNUs-WjvSIcPDIHr1Qgtqm


In [1]:
# Robotics arena runtime diagnostics. No package installs or training.
import importlib.metadata
import json
import platform
import shutil
import subprocess

versions = {}
for package in ('mujoco', 'mujoco-mjx', 'jax', 'jaxlib', 'numpy'):
    try:
        versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        versions[package] = None
print(json.dumps({'python': platform.python_version(), 'system': platform.system(), 'packages': versions}, indent=2))
if shutil.which('nvidia-smi'):
    print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], text=True))
else:
    print('No nvidia-smi executable')


{
  "python": "3.13.15",
  "system": "Linux",
  "packages": {
    "mujoco": null,
    "mujoco-mjx": null,
    "jax": "0.11.1",
    "jaxlib": "0.11.1",
    "numpy": "2.1.3"
  }
}
NVIDIA A100-SXM4-40GB, 40960 MiB, 580.82.07


In [2]:
# Pinned dependencies for Linux native MuJoCo checks.
import os
import subprocess
import sys
os.environ['MUJOCO_GL'] = 'egl'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'mujoco==3.12.0', 'gymnasium==1.3.0', 'numpy>=2.1,<3'], check=True)
import importlib.metadata as metadata
import json
print(json.dumps({name: metadata.version(name) for name in ('mujoco','gymnasium','numpy')}, indent=2))


{
  "mujoco": "3.12.0",
  "gymnasium": "1.3.0",
  "numpy": "2.1.3"
}


In [3]:
# Optional version-matched Warp compatibility inspection.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'mujoco-mjx[warp]==3.12.0'], check=True)
import dataclasses
import mujoco
from mujoco import mjx
import warp as wp
from mujoco.mjx.third_party.mujoco_warp._src import types as bundled_warp_types
fields = {field.name for field in dataclasses.fields(bundled_warp_types.Model)}
print(json.dumps({'mujoco': mujoco.__version__, 'mujoco_mjx': metadata.version('mujoco-mjx'), 'warp_lang': metadata.version('warp-lang'), 'pair_adhesion_field': 'pair_adhesion' in fields, 'geom_adhesion_field': 'geom_adhesion' in fields, 'flex_bending_field': 'flex_bending' in fields, 'gpu_devices': [str(device) for device in wp.get_cuda_devices()]}, indent=2))


Warp 1.16.0 initialized:
   CUDA Toolkit 12.9, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA A100-SXM4-40GB" (39 GiB, sm_80, mempool enabled)
   Kernel cache:
     /root/.cache/warp/1.16.0
{
  "mujoco": "3.12.0",
  "mujoco_mjx": "3.12.0",
  "warp_lang": "1.16.0",
  "pair_adhesion_field": false,
  "geom_adhesion_field": false,
  "flex_bending_field": true,
  "gpu_devices": [
    "cuda:0"
  ]
}


In [4]:
# Native adhesion coupon plus an explicit Warp compatibility gate.
import numpy as np
from pathlib import Path
from PIL import Image
from mujoco.mjx.third_party import mujoco_warp as bundled_warp
coupon_xml = '''<mujoco><option timestep=".0001" gravity="0 0 0" solver="Newton" integrator="implicitfast"/><worldbody><light pos="0 0 1"/><geom name="floor" type="plane" size=".1 .1 .01" rgba=".5 .5 .5 1"/><body name="bead" pos="0 0 .01"><freejoint/><geom name="bead_geom" type="sphere" size=".01" mass=".01" rgba=".8 .2 .1 1"/></body></worldbody><contact><pair name="bond" geom1="floor" geom2="bead_geom" adhesion="{adh}" condim="3" margin="0" gap=".001" solref=".0004 1" solimp=".99 .999 .00001"/></contact></mujoco>'''
adhesion_probe = {'native': {}}
for adhesion in (0.0, 0.1):
    coupon_model = mujoco.MjModel.from_xml_string(coupon_xml.format(adh=adhesion))
    coupon_data = mujoco.MjData(coupon_model)
    coupon_data.xfrc_applied[1, 2] = 0.05
    for _ in range(1000):
        mujoco.mj_step(coupon_model, coupon_data)
    adhesion_probe['native'][str(adhesion)] = {'height_m': float(coupon_data.qpos[2]), 'vertical_speed_m_s': float(coupon_data.qvel[2]), 'contacts': int(coupon_data.ncon), 'warnings': sum(int(w.number) for w in coupon_data.warning)}
with wp.ScopedDevice('cuda:0'):
    coupon_warp_model = bundled_warp.put_model(coupon_model)
    adhesion_probe['warp'] = {'model_conversion': 'accepted', 'pair_adhesion_field': hasattr(coupon_warp_model, 'pair_adhesion'), 'geom_adhesion_field': hasattr(coupon_warp_model, 'geom_adhesion'), 'full_arena_compatibility': 'blocked: contact adhesion fields are absent; Python bond updates also require a separate device implementation', 'simulation_steps': 0}
output_dir = Path('/content/robotics_runtime_checks')
output_dir.mkdir(exist_ok=True)
with mujoco.Renderer(coupon_model, height=240, width=320) as renderer:
    camera = mujoco.MjvCamera()
    camera.lookat[:] = [0,0,0.01]
    camera.distance = 0.18
    camera.azimuth = 45
    camera.elevation = -30
    renderer.update_scene(coupon_data, camera=camera)
    pixels = renderer.render()
    from OpenGL.GL import glGetString, GL_RENDERER
    gpu_renderer = glGetString(GL_RENDERER).decode()
    Image.fromarray(pixels).save(output_dir/'adhesion_probe.png')
    adhesion_probe['offscreen_render'] = {'shape': list(pixels.shape), 'pixel_std': float(pixels.std()), 'gl_renderer': gpu_renderer, 'backend': 'EGL', 'physics_backend': 'native MuJoCo CPU'}
assert adhesion_probe['native']['0.0']['height_m'] > 0.03
assert abs(adhesion_probe['native']['0.1']['height_m'] - 0.01) < 1e-6
assert adhesion_probe['offscreen_render']['pixel_std'] > 1
(output_dir/'adhesion_compatibility.json').write_text(json.dumps(adhesion_probe, indent=2))
print(json.dumps(adhesion_probe, indent=2))


{
  "native": {
    "0.0": {
      "height_m": 0.03502500000000002,
      "vertical_speed_m_s": 0.5000000000000003,
      "contacts": 0,
      "warnings": 0
    },
    "0.1": {
      "height_m": 0.010000008146106378,
      "vertical_speed_m_s": 2.8231027084480157e-14,
      "contacts": 1,
      "warnings": 0
    }
  },
  "warp": {
    "model_conversion": "accepted",
    "pair_adhesion_field": false,
    "geom_adhesion_field": false,
    "full_arena_compatibility": "blocked: contact adhesion fields are absent; Python bond updates also require a separate device implementation",
    "simulation_steps": 0
  },
  "offscreen_render": {
    "shape": [
      240,
      320,
      3
    ],
    "pixel_std": 60.26860954976984,
    "gl_renderer": "llvmpipe (LLVM 15.0.7, 256 bits)",
    "backend": "EGL",
    "physics_backend": "native MuJoCo CPU"
  }
}


In [6]:
# Inspect EGL vendor registration without changing the driver.
from pathlib import Path
import ctypes.util
print(json.dumps({'egl_vendor_files': [str(p) for p in Path('/usr/share/glvnd/egl_vendor.d').glob('*.json')], 'nvidia_egl_library': ctypes.util.find_library('EGL_nvidia')}, indent=2))
# Inspect EGL vendor registration without changing the driver.
from pathlib import Path
import ctypes.util
print(json.dumps({'egl_vendor_files': [str(p) for p in Path('/usr/share/glvnd/egl_vendor.d').glob('*.json')], 'nvidia_egl_library': ctypes.util.find_library('EGL_nvidia')}, indent=2))
import glob
print(json.dumps({'nvidia_egl_candidates': glob.glob('/usr/lib64-nvidia/*EGL*') + glob.glob('/usr/lib/x86_64-linux-gnu/*EGL*nvidia*')}, indent=2))


{
  "egl_vendor_files": [
    "/usr/share/glvnd/egl_vendor.d/50_mesa.json"
  ],
  "nvidia_egl_library": null
}
{
  "egl_vendor_files": [
    "/usr/share/glvnd/egl_vendor.d/50_mesa.json"
  ],
  "nvidia_egl_library": null
}
{
  "nvidia_egl_candidates": [
    "/usr/lib64-nvidia/libEGL_nvidia.so.0",
    "/usr/lib64-nvidia/libEGL.so",
    "/usr/lib64-nvidia/libEGL.so.1.1.0",
    "/usr/lib64-nvidia/libEGL_nvidia.so.580.82.07",
    "/usr/lib64-nvidia/libEGL.so.1"
  ]
}


In [7]:
# Register Colab's existing NVIDIA EGL driver, following the official MuJoCo tutorial.
# Use a fresh subprocess because the notebook already initialized Mesa EGL.
nvidia_icd = Path('/usr/share/glvnd/egl_vendor.d/10_nvidia.json')
if not nvidia_icd.exists():
    nvidia_icd.write_text(json.dumps({'file_format_version':'1.0.0','ICD':{'library_path':'/usr/lib64-nvidia/libEGL_nvidia.so.0'}}))
os.environ['MUJOCO_GL'] = 'egl'
os.environ['__EGL_VENDOR_LIBRARY_FILENAMES'] = str(nvidia_icd)
os.environ['LD_LIBRARY_PATH'] = '/usr/lib64-nvidia:' + os.environ.get('LD_LIBRARY_PATH','')
gpu_render_source = '''import json
from pathlib import Path
import mujoco
from PIL import Image
from OpenGL.GL import glGetString, GL_RENDERER
model = mujoco.MjModel.from_xml_string('<mujoco><worldbody><light pos="0 0 1"/><geom type="plane" size=".1 .1 .01"/><body pos="0 0 .01"><freejoint/><geom type="sphere" size=".01" mass=".01" rgba=".8 .2 .1 1"/></body></worldbody></mujoco>')
data = mujoco.MjData(model)
mujoco.mj_forward(model,data)
with mujoco.Renderer(model,height=240,width=320) as renderer:
    camera = mujoco.MjvCamera()
    camera.lookat[:] = [0,0,.01]
    camera.distance = .18
    camera.azimuth = 45
    camera.elevation = -30
    renderer.update_scene(data,camera=camera)
    image = renderer.render()
    result = {'backend':'EGL','gl_renderer':glGetString(GL_RENDERER).decode(),'shape':list(image.shape),'pixel_std':float(image.std()),'physics_backend':'native MuJoCo CPU'}
    assert 'NVIDIA' in result['gl_renderer'], result
    assert result['pixel_std'] > 1
    Image.fromarray(image).save('/content/robotics_runtime_checks/nvidia_egl_probe.png')
Path('/content/robotics_runtime_checks/nvidia_egl_probe.json').write_text(json.dumps(result,indent=2))
print(json.dumps(result,indent=2))
'''
render_process = subprocess.run([sys.executable,'-c',gpu_render_source],env=os.environ.copy(),capture_output=True,text=True,timeout=60)
print(render_process.stdout)
print(render_process.stderr)
render_process.check_returncode()


{
  "backend": "EGL",
  "gl_renderer": "NVIDIA A100-SXM4-40GB/PCIe/SSE2",
  "shape": [
    240,
    320,
    3
  ],
  "pixel_std": 60.27006239334972,
  "physics_backend": "native MuJoCo CPU"
}


In [8]:
# Validate the public migration branch in an isolated directory.
import time
repo_dir = Path('/content/robotics_mujoco_dc77f2a')
if not repo_dir.exists():
    subprocess.run(['git','clone','--depth','1','--branch','feat/mujoco-physics','https://github.com/ghandhitechnology/robotics-challenge-arena.git',str(repo_dir)],check=True)
tested_commit = subprocess.check_output(['git','rev-parse','HEAD'],cwd=repo_dir,text=True).strip()
print('TESTED_COMMIT',tested_commit,flush=True)
commands = [
    [sys.executable,'-m','arena_mujoco','run','--seconds','.02','--output','/content/native_report.json'],
    [sys.executable,'-m','unittest','discover','-s','tests/mujoco','-p','test_env.py','-v'],
    [sys.executable,'-m','arena_mujoco','render','--seconds','0','--output','/content/arena.png'],
]
execution_report = {'commit':tested_commit,'platform':'Colab Linux A100','physics_backend':'native MuJoCo CPU','render_backend':'NVIDIA EGL','commands':[]}
for command in commands:
    start = time.monotonic()
    try:
        result = subprocess.run(command,cwd=repo_dir,env=os.environ.copy(),capture_output=True,text=True,timeout=240)
        record = {'command':command[1:],'returncode':result.returncode,'elapsed_s':round(time.monotonic()-start,3),'stdout':result.stdout[-20000:],'stderr':result.stderr[-20000:]}
    except subprocess.TimeoutExpired:
        record = {'command':command[1:],'returncode':124,'elapsed_s':round(time.monotonic()-start,3),'error':'240 second timeout'}
    execution_report['commands'].append(record)
    print(json.dumps(record,indent=2),flush=True)
execution_report['passed'] = all(item['returncode']==0 for item in execution_report['commands'])
Path('/content/robotics_runtime_checks/native_execution.json').write_text(json.dumps(execution_report,indent=2))
print('ALL_NATIVE_CHECKS_PASSED',execution_report['passed'],flush=True)


TESTED_COMMIT dc77f2aeae639e3120ece2857d74bcfb3bec24d5
{
  "command": [
    "-m",
    "arena_mujoco",
    "run",
    "--seconds",
    ".02",
    "--output",
    "/content/native_report.json"
  ],
  "returncode": 0,
  "elapsed_s": 16.877,
  "stdout": "{\n  \"time_s\": 0.01999999999999999,\n  \"contacts\": 1425,\n  \"max_abs_qvel\": 0.0003611203489392055,\n  \"warnings\": 0,\n  \"tape_bond_count\": 1323,\n  \"tape_bonded_count\": 1323,\n  \"tape_bonded_area_m2\": 0.08602000000000004,\n  \"tape_damage_fraction\": 0.0,\n  \"tape_wear_fraction\": 0.0,\n  \"tape_max_opening_m\": 0.0,\n  \"tape_max_slip_m\": 2.6175987629190863e-06,\n  \"tape_damage_energy_proxy_j\": 0.0,\n  \"tape_contact_work_proxy_j\": 9.9318882220322e-07,\n  \"tape_reach_limited_releases\": 0,\n  \"tape_orientation_releases\": 0,\n  \"tape_rebond_count\": 0,\n  \"wall_seconds\": 8.083107351000308,\n  \"seed\": 0,\n  \"mujoco_version\": \"3.12.0\",\n  \"backend\": \"native CPU\"\n}\n",
  "stderr": ""
}
{
  "command": [
    

In [9]:
# Persist the tested commit, native metrics, and rendered arena preview.
from IPython.display import display
import hashlib
arena_image = Image.open('/content/arena.png')
combined_report = {'commit':tested_commit,'execution':execution_report,'native':json.loads(Path('/content/native_report.json').read_text()),'adhesion_gate':adhesion_probe,'nvidia_egl':json.loads(Path('/content/robotics_runtime_checks/nvidia_egl_probe.json').read_text()),'arena_image':{'dimensions':list(arena_image.size),'pixel_std':float(np.asarray(arena_image).std()),'sha256':hashlib.sha256(Path('/content/arena.png').read_bytes()).hexdigest()}}
assert combined_report['arena_image']['pixel_std'] > 1
Path('/content/robotics_runtime_checks/combined_report.json').write_text(json.dumps(combined_report,indent=2))
print(json.dumps(combined_report,indent=2))
display(arena_image)


{
  "commit": "dc77f2aeae639e3120ece2857d74bcfb3bec24d5",
  "execution": {
    "commit": "dc77f2aeae639e3120ece2857d74bcfb3bec24d5",
    "platform": "Colab Linux A100",
    "physics_backend": "native MuJoCo CPU",
    "render_backend": "NVIDIA EGL",
    "commands": [
      {
        "command": [
          "-m",
          "arena_mujoco",
          "run",
          "--seconds",
          ".02",
          "--output",
          "/content/native_report.json"
        ],
        "returncode": 0,
        "elapsed_s": 16.877,
        "stdout": "{\n  \"time_s\": 0.01999999999999999,\n  \"contacts\": 1425,\n  \"max_abs_qvel\": 0.0003611203489392055,\n  \"warnings\": 0,\n  \"tape_bond_count\": 1323,\n  \"tape_bonded_count\": 1323,\n  \"tape_bonded_area_m2\": 0.08602000000000004,\n  \"tape_damage_fraction\": 0.0,\n  \"tape_wear_fraction\": 0.0,\n  \"tape_max_opening_m\": 0.0,\n  \"tape_max_slip_m\": 2.6175987629190863e-06,\n  \"tape_damage_energy_proxy_j\": 0.0,\n  \"tape_contact_work_proxy_j\": 9.9

In [10]:
# Validate the final source commit with the complete native MuJoCo suite.
import time
final_commit = 'd8c20fd6d9a81cb34986356cd0b222109f94d0f5'
final_repo_dir = Path('/content/robotics_mujoco_d8c20fd')
if not final_repo_dir.exists():
    subprocess.run(['git','clone','--depth','1','--branch','feat/mujoco-physics','https://github.com/ghandhitechnology/robotics-challenge-arena.git',str(final_repo_dir)],check=True)
subprocess.run(['git','fetch','--depth','1','origin',final_commit],cwd=final_repo_dir,check=True)
subprocess.run(['git','checkout','--detach',final_commit],cwd=final_repo_dir,check=True)
final_tested_commit = subprocess.check_output(['git','rev-parse','HEAD'],cwd=final_repo_dir,text=True).strip()
assert final_tested_commit == final_commit
print('FINAL_TESTED_COMMIT',final_tested_commit,flush=True)
final_commands = [
    [sys.executable,'-m','unittest','discover','-s','tests/mujoco','-p','test_*.py','-v'],
    [sys.executable,'-m','arena_mujoco','run','--seconds','.02','--output','/content/native_report_final.json'],
    [sys.executable,'-m','arena_mujoco','render','--seconds','0','--output','/content/arena_final.png'],
]
final_execution = {'commit':final_tested_commit,'platform':'Colab Linux A100','physics_backend':'native MuJoCo CPU','render_backend':'NVIDIA EGL','commands':[]}
for command in final_commands:
    start = time.monotonic()
    try:
        result = subprocess.run(command,cwd=final_repo_dir,env=os.environ.copy(),capture_output=True,text=True,timeout=240)
        record = {'command':command[1:],'returncode':result.returncode,'elapsed_s':round(time.monotonic()-start,3),'stdout':result.stdout[-20000:],'stderr':result.stderr[-20000:]}
    except subprocess.TimeoutExpired:
        record = {'command':command[1:],'returncode':124,'elapsed_s':round(time.monotonic()-start,3),'error':'240 second timeout'}
    final_execution['commands'].append(record)
    print(json.dumps(record,indent=2),flush=True)
final_execution['passed'] = all(item['returncode']==0 for item in final_execution['commands'])
Path('/content/robotics_runtime_checks/final_execution.json').write_text(json.dumps(final_execution,indent=2))
assert final_execution['passed'], final_execution
final_arena_image = Image.open('/content/arena_final.png')
final_report = {'commit':final_tested_commit,'execution':final_execution,'native':json.loads(Path('/content/native_report_final.json').read_text()),'nvidia_egl':json.loads(Path('/content/robotics_runtime_checks/nvidia_egl_probe.json').read_text()),'arena_image':{'dimensions':list(final_arena_image.size),'pixel_std':float(np.asarray(final_arena_image).std()),'sha256':hashlib.sha256(Path('/content/arena_final.png').read_bytes()).hexdigest()}}
assert final_report['arena_image']['pixel_std'] > 1
Path('/content/robotics_runtime_checks/final_report.json').write_text(json.dumps(final_report,indent=2))
print('FINAL_REPORT',json.dumps(final_report,indent=2),flush=True)


FINAL_TESTED_COMMIT d8c20fd6d9a81cb34986356cd0b222109f94d0f5
{
  "command": [
    "-m",
    "unittest",
    "discover",
    "-s",
    "tests/mujoco",
    "-p",
    "test_*.py",
    "-v"
  ],
  "returncode": 0,
  "elapsed_s": 60.708,
  "stdout": "",
  "stderr": "test_blank_template_retains_base_and_rejects_invalid_measurements (test_calibration.CalibrationTests.test_blank_template_retains_base_and_rejects_invalid_measurements) ... ok\ntest_friction_density_and_linear_modulus (test_calibration.CalibrationTests.test_friction_density_and_linear_modulus) ... ok\ntest_lab_holes_and_peel_angle_units (test_calibration.CalibrationTests.test_lab_holes_and_peel_angle_units) ... ok\ntest_flex_step_and_checkpoint (test_env.EnvironmentTests.test_flex_step_and_checkpoint) ... ok\ntest_gym_contract (test_env.EnvironmentTests.test_gym_contract) ... ok\ntest_seeded_reset_and_delayed_noisy_action_replay (test_env.EnvironmentTests.test_seeded_reset_and_delayed_noisy_action_replay) ... ok\ntest_success_and